# Scaling the *number of logical qubits* — non-Clifford circuits on 3, 4, and 5 patches (d=3)

The other notebooks in `8.Scaling` pushed the **code distance** `d`. This one holds `d=3` fixed and
pushes the **other** axis: how many *logical qubits* can we entangle with genuinely non-Clifford
(T-gate) circuits before runtime or accuracy gives out?

The MPS engine (`scaling_engine.jl`) already parameterises the patch count, and its transversal CNOT,
Hadamard, and frame/reference bookkeeping are patch-agnostic. What was hard-wired to two logical qubits
was only the *driver*: the circuit runner, the S/T teleportation gadget (which used a fixed patch-3
ancilla), and the exact reference. `multiqubit_runner.jl` lifts those to arbitrary `Q`, changing
nothing in the validated engine.

## §1 — The generalisation

- **`run_circuit_mq(c, Q, circuit)`** prepares data patches `1..Q` in `|0⟩_L` and reuses a **single**
  magic-ancilla patch (index `Q+1`) for every S/T teleportation — prepared fresh and measured out each
  time, so one scratch patch serves the whole circuit. The code must be built with `npatch ≥ Q+1`.
- **`teleport_{S,T}_at!`** are the engine's gadgets with the ancilla patch passed explicitly.
- **`logical_expect(M, [(patch,L)…])`** reads a frame-corrected Pauli-string correlator over *any* set
  of patches (it reuses the engine's per-patch operator/phase/frame-sign builder).
- **`ref_state_mq` / `refN`** give the exact `2^Q` state vector and its correlators as ground truth.

Two choices are held fixed, deliberately: patches are **side-by-side** (so a transversal CNOT between
non-adjacent logical qubits *visibly* spans the MPS chain — the dominant cost here), and entangling
gates are **transversal CNOTs** (no lattice surgery).

In [ ]:
include("multiqubit_runner.jl");

## §2 — The test circuit: a T-decorated GHZ

For each `Q` we run a circuit that is entangling *and* non-Clifford, with a known exact reference:

$$\texttt{H}_1,\; \texttt{T}_1,\; \texttt{CNOT}_{1\to2},\,\texttt{CNOT}_{2\to3},\,\dots,\texttt{CNOT}_{Q-1\to Q},\; \texttt{T}_Q$$

The `H,T` on qubit 1 injects a magic (`|A⟩ = T|+⟩`) amplitude; the CNOT chain spreads it into a
GHZ-like entangled state across all `Q` logical qubits; the closing `T` on qubit `Q` adds a second
non-Clifford rotation at the far end of the chain (forcing a transversal CNOT to the ancilla across the
*whole* register — the worst case for bond dimension). Two T-gates keep the teleportation cost bounded
while still leaving the state outside the Clifford group.

In [ ]:
tghz(Q) = vcat(Any[(:H,1),(:T,1)], Any[(:CNOT,i,i+1) for i in 1:Q-1], Any[(:T,Q)])
println("Q=3 circuit: ", tghz(3))

## §3 — Regression: the generalisation reproduces the 2-qubit engine

Before trusting `Q>2`, confirm that at `Q=2` the general reference agrees with the engine's original
`ref_state`, and that the MPS run matches the exact state vector to machine precision (this is the same
magic-Bell state validated in `example_scaling_analysis.ipynb`, now driven through `run_circuit_mq`).

In [ ]:
bell = Any[(:H,1),(:T,1),(:CNOT,1,2)]     # the single-T magic Bell from example_scaling_analysis.ipynb
println("general ref == engine ref_state:  ", isapprox(ref_state_mq(2, bell), ref_state(bell); atol=1e-12))
set_precision!(cutoff=1e-6, maxdim=512)
c = build_code(3; npatch=3); Random.seed!(21)
M = run_circuit_mq(c, 2, bell; ec=false, R=3, C=2, B=2, use_AD=true)
ψ = ref_state_mq(2, bell)
@printf("  ⟨ZZ⟩ MPS=%.4f exact=%.4f    ⟨XX⟩ MPS=%.4f exact=%.4f   (expect 1.000 / 0.707)\n",
        logical_expect(M,[(1,:Z),(2,:Z)]), refN(ψ,2,[(1,:Z),(2,:Z)]),
        logical_expect(M,[(1,:X),(2,:X)]), refN(ψ,2,[(1,:X),(2,:X)]))

## §4 — Scaling the logical-qubit count: Q = 2, 3, 4, 5

For each `Q` we build a `Q+1`-patch d=3 code, run the T-GHZ circuit, and compare **every** single-qubit
`⟨Z_i⟩`/`⟨X_i⟩`, the neighbour `⟨Z_iZ_{i+1}⟩`, and the full `⟨Z^{⊗Q}⟩`/`⟨X^{⊗Q}⟩` against the exact
state vector — reporting the worst error, the peak bond dimension, and the wall-clock time.

Measured (d=3, `ec=false`, `maxdim=512`; worst error over all the Pauli strings above):

| Q | patches | MPS sites | χ peak | cap_hits | max error | time |
|---|---|---|---|---|---|---|
| 2 | 3 | 51  | 32 | 0 | 2·10⁻¹⁴ | 29 s* |
| 3 | 4 | 68  | 32 | 0 | 7·10⁻¹⁵ | 3.8 s |
| 4 | 5 | 85  | 32 | 0 | 7·10⁻¹⁵ | 5.7 s |
| 5 | 6 | 102 | 32 | 0 | 4·10⁻¹⁴ | 6.7 s |

<sub>*Q=2 includes one-off JIT compilation.</sub>

Two clean findings:

- **Accuracy is machine precision at every `Q`.** The engine is *correct* for multi-qubit non-Clifford
  circuits, not just two — every single-qubit, neighbour, and full-register correlator tracks the exact
  `2^Q` state vector to ~10⁻¹⁴. The generalisation touched only the driver; the physics was already
  qubit-agnostic.
- **Bond dimension does *not* grow with `Q` here — it is flat at 32.** That is not luck: a GHZ-type
  state has *low* entanglement (Schmidt rank 2 across any cut), so the peak χ is set by each patch's own
  d=3 encoding (χ≈32), not by the inter-qubit entanglement. Runtime therefore grows only ~linearly with
  `Q` (more MPS sites to sweep), not exponentially. **On this class of circuit, Q=5 is easy and Q=10+
  would be routine.**

So for *low-entanglement* non-Clifford circuits the logical-qubit count is essentially free. That
raises the real question — what makes multi-qubit *hard*? The next cell answers it.

In [ ]:
set_precision!(cutoff=1e-6, maxdim=512)
println("Q  patches  sites  χmax  cap_hits  maxErr     time(s)")
for Q in (2, 3, 4, 5)
    circ = tghz(Q)
    c = build_code(3; npatch=Q+1); Random.seed!(21)
    t = @elapsed M = run_circuit_mq(c, Q, circ; ec=false, R=3, C=2, B=2, use_AD=true)
    ψ = ref_state_mq(Q, circ)
    specs = Vector{Any}[]
    for i in 1:Q; push!(specs, [(i,:Z)]); push!(specs, [(i,:X)]); end
    for i in 1:Q-1; push!(specs, [(i,:Z),(i+1,:Z)]); end
    push!(specs, [(i,:Z) for i in 1:Q]); push!(specs, [(i,:X) for i in 1:Q])
    err = maximum(abs(logical_expect(M,s) - refN(ψ,Q,s)) for s in specs)
    @printf("%d  %-7d  %-5d  %-4d  %-8d  %.1e   %6.1f\n",
            Q, c.npatch, length(c.sites), CHIMAX[], CAP_HITS[], err, t)
end

## §4b — What actually limits `Q`: entanglement, not qubit count

To show the boundary is *entanglement* rather than the number of qubits, we run a deliberately
**high-entanglement** circuit at the same `d=3`: Hadamard every qubit, then entangle each qubit `i`
with its mirror `Q+1-i` — so **every** CNOT crosses the centre of the register — and add two T-gates.
Now the entanglement across the middle cut grows with `Q` (roughly `2^{Q/2}` Bell pairs must be carried
there), so the bond dimension must climb.

Measured (d=3, `ec=false`):

| Q | χ peak | cap_hits | max error | time |
|---|---|---|---|---|
| 2 | 32  | 0 | 4·10⁻¹⁵ | 33 s* |
| 4 | 64  | 0 | 3·10⁻¹⁴ | 7.5 s |
| 6 | 128 | 0 | 2·10⁻¹⁴ | 14 s |

<sub>*Q=2 includes JIT compilation.</sub>

Same code distance, same engine, same machine-precision accuracy — but now **χ doubles every two
qubits** (32 → 64 → 128, i.e. the `2^{Q/2}` Bell pairs the centre cut must carry), and runtime rises
with it. Compare to §4's T-GHZ, where χ never budged from 32. This is the honest boundary: what the MPS
cannot afford is not *more logical qubits* per se, but the *entanglement* a circuit spreads across the
side-by-side chain. Low-rank states (GHZ, product-ish, lightly-entangled) scale to many qubits for
free; volume-law entanglers hit the same exponential wall as the distance axis.

In [ ]:
xent(Q) = vcat(Any[(:H,i) for i in 1:(Q÷2)],             # H the controls (left half) only …
               Any[(:CNOT, i, Q+1-i) for i in 1:(Q÷2)],   # … so each CNOT makes a Bell pair across centre
               Any[(:T,1),(:T,Q)])
set_precision!(cutoff=1e-6, maxdim=2048)
println("Q  χpeak  cap_hits  maxErr    time(s)   (centre-crossing entangler — χ grows with Q)")
for Q in (2, 4, 6)
    circ = xent(Q); c = build_code(3; npatch=Q+1); Random.seed!(21)
    t = @elapsed M = run_circuit_mq(c, Q, circ; ec=false, R=3, C=2, B=2, use_AD=true)
    ψ = ref_state_mq(Q, circ)
    specs = Vector{Any}[]
    for i in 1:Q; push!(specs, [(i,:Z)]); push!(specs, [(i,:X)]); end
    push!(specs, [(i,:Z) for i in 1:Q]); push!(specs, [(i,:X) for i in 1:Q])
    err = maximum(abs(logical_expect(M,s) - refN(ψ,Q,s)) for s in specs)
    @printf("%d  %-5d  %-8d  %.1e   %6.1f\n", Q, CHIMAX[], CAP_HITS[], err, t)
end

## §5 — A *deeper* non-Clifford circuit

The T-GHZ was deliberately simple. Here is a genuinely deeper Clifford+T circuit on `Q` qubits — a GHZ
core dressed with two non-Clifford layers and a basis rotation:

1. **H** on qubit 1, then a **CNOT chain** `1→2→…→Q` — a GHZ state entangling *all* qubits;
2. **T** on every qubit — a magic amplitude on each;
3. **S** on the odd qubits — a phase-varied layer;
4. **H** on every qubit — rotate the basis so the phase structure shows up as a non-uniform distribution;
5. **T** on every qubit — a second magic layer.

That is ~`2Q` teleported non-Clifford gates on top of a fully-entangling GHZ core, and the state it makes
has no simple closed form. We run it at Q = 2, 3, 4 and check *every* single-qubit
`⟨X_i⟩/⟨Y_i⟩/⟨Z_i⟩`, neighbour `⟨Z_iZ_{i+1}⟩/⟨X_iX_{i+1}⟩`, and full-register correlator against the
exact `2^Q` state vector.

Measured (d=3, `ec=false`, `maxdim=1024`):

| Q | gates | χ peak | cap_hits | max error | time |
|---|---|---|---|---|---|
| 2 | 9  | 32  | 0 | 5·10⁻¹⁵ | 40 s* |
| 3 | 14 | 64  | 0 | 2·10⁻¹⁴ | 18 s |
| 4 | 18 | 128 | 0 | 2·10⁻¹⁴ | 28 s |

<sub>*Q=2 includes JIT compilation.</sub>

**Machine-precision agreement with the exact state at every `Q`**, at ~2× the gate count of the
T-GHZ. The bond dimension now *does* climb (32 → 64 → 128) — more than the flat χ=32 of §4's single-T
GHZ — because the GHZ core spreads a full bit of entanglement across every cut and the T/S/H dressing
adds structure on top. It is still perfectly tractable (χ=128, ~30 s at Q=4); the growth just confirms
§4b's lesson — cost tracks the entanglement the circuit builds, and this circuit builds more.

In [ ]:
function deep_circ(Q)
    c = Any[]
    push!(c,(:H,1))
    for i in 1:(Q-1); push!(c,(:CNOT,i,i+1)); end       # GHZ: entangle all qubits
    for i in 1:Q;     push!(c,(:T,i)); end              # magic on each
    for i in 1:2:Q;   push!(c,(:S,i)); end              # S on odd qubits
    for i in 1:Q;     push!(c,(:H,i)); end              # rotate basis → non-uniform distribution
    for i in 1:Q;     push!(c,(:T,i)); end              # second magic layer
    c
end
allspecs(Q) = begin
    s = Vector{Any}[]
    for i in 1:Q, P in (:X,:Y,:Z); push!(s,[(i,P)]); end
    for i in 1:Q-1; push!(s,[(i,:Z),(i+1,:Z)]); push!(s,[(i,:X),(i+1,:X)]); end
    push!(s,[(i,:Z) for i in 1:Q]); push!(s,[(i,:X) for i in 1:Q]); s
end
set_precision!(cutoff=1e-6, maxdim=1024)
println("Q  gates  χpeak  cap_hits  maxErr    time(s)")
for Q in (2,3,4)
    circ = deep_circ(Q); c = build_code(3; npatch=Q+1); Random.seed!(7)
    t = @elapsed M = run_circuit_mq(c, Q, circ; ec=false, R=3, C=2, B=2, use_AD=true)
    ψ = ref_state_mq(Q, circ)
    err = maximum(abs(logical_expect(M,s) - refN(ψ,Q,s)) for s in allspecs(Q))
    @printf("%d  %-5d  %-5d  %-8d  %.1e   %5.1f\n", Q, length(circ), CHIMAX[], CAP_HITS[], err, t)
end

## §6 — Observables: what state did we actually make?

Accuracy tables prove the simulator is *right*, but they do not show *what state it produced*. Here we
characterise the Q=3 deep-circuit output four ways, reading each from the MPS (frame-corrected) and, for
the global pictures, from the exact reference:

- **Single-qubit Bloch vectors** `(⟨X_i⟩,⟨Y_i⟩,⟨Z_i⟩)` — where each logical qubit's *marginal* sits on
  its Bloch sphere. For a strongly-entangled register these shrink toward the origin (the marginals are
  mixed), so a length-zero Bloch vector is itself evidence of entanglement.
- **Two-qubit correlators** `⟨Z_iZ_j⟩, ⟨X_iX_j⟩` — the correlation structure the GHZ core built.
- **Computational-basis distribution** `P(b) = |⟨b|ψ⟩|²` — the "shape" of the state over all `2^Q`
  outcomes, and (as we'll see) the clearest certificate of non-Cliffordness.
- **Bipartite entanglement entropy** across each cut — how genuinely entangled the register is (and
  hence why the MPS bond dimension is what it is).

Measured for the Q=3 deep circuit (MPS vs exact — every entry agrees to ~10⁻¹⁴):

| observable | value | what it tells us |
|---|---|---|
| `⟨X_i⟩=⟨Y_i⟩=⟨Z_i⟩` = 0, all `i` | Bloch length 0 | every single-qubit marginal is **maximally mixed** |
| `⟨Z_iZ_j⟩` = 0, `⟨X_iX_j⟩` = 0.5, all pairs | — | correlations live in X, not Z — the GHZ signature in the rotated basis |
| `S` across **every** cut = **1.000 bit** | — | the register is **maximally (GHZ-class) entangled** — all three qubits in one block |
| `P(b)` = **0.2134 / 0.0366** by bit-parity | `(1 ± 1/√2)/8` | the **non-Clifford fingerprint** |

The picture these paint is precise and it is exactly what a magic computation should make:

- **Locally you see nothing.** Every single-qubit Bloch vector is zero — the marginals are maximally
  mixed — and the entropy is a full bit across every cut. All the state's content is in the *global*
  entanglement, not in any one qubit. That is the hallmark of a GHZ-class state.
- **The distribution proves it is non-Clifford.** The computational-basis probabilities take just two
  values, `(1 + 1/√2)/8 ≈ 0.213` and `(1 - 1/√2)/8 ≈ 0.037`, split by the parity of the bitstring. The
  `1/√2 = cos(π/4)` is the **T-gate's phase made visible**: a stabiliser (Clifford-only) state can only
  have *dyadic* probabilities (multiples of `1/2^k`), never an irrational like `1/√2`. So this single
  observable is a certificate that the engine really did produce a magic state, not a Clifford one.
- **And the MPS got all of it right** — every Bloch component, correlator, and (via the validated
  Pauli strings) the whole state, to machine precision at d=3.

In [ ]:
Q = 3; circ = deep_circ(Q); c = build_code(3; npatch=Q+1); Random.seed!(7)
M = run_circuit_mq(c, Q, circ; ec=false, R=3, C=2, B=2, use_AD=true)
ψ = ref_state_mq(Q, circ)

println("single-qubit Bloch  (MPS / exact):")
for i in 1:Q
    @printf("  qubit %d:  X=% .3f/% .3f   Y=% .3f/% .3f   Z=% .3f/% .3f\n", i,
        logical_expect(M,[(i,:X)]), refN(ψ,Q,[(i,:X)]),
        logical_expect(M,[(i,:Y)]), refN(ψ,Q,[(i,:Y)]),
        logical_expect(M,[(i,:Z)]), refN(ψ,Q,[(i,:Z)]))
end
println("two-qubit correlators  (MPS / exact):")
for (a,b) in [(1,2),(2,3),(1,3)]
    @printf("  ⟨Z%d Z%d⟩=% .3f/% .3f    ⟨X%d X%d⟩=% .3f/% .3f\n", a,b,
        logical_expect(M,[(a,:Z),(b,:Z)]), refN(ψ,Q,[(a,:Z),(b,:Z)]),
        a,b, logical_expect(M,[(a,:X),(b,:X)]), refN(ψ,Q,[(a,:X),(b,:X)]))
end
println("computational-basis probabilities |q1q2q3⟩ (exact, P>0.001):")
for b in 0:2^Q-1
    p = abs2(ψ[b+1]); p > 1e-3 && @printf("  |%s⟩ : %.4f\n", string(b, base=2, pad=Q), p)
end
using LinearAlgebra
_ent(ψ,Q,k) = (s = svdvals(reshape(ψ, 2^(Q-k), 2^k)); p = (s.^2); p = p[p .> 1e-14]; -sum(p .* log2.(p)))
println("bipartite entanglement entropy (exact):")
for k in 1:Q-1; @printf("  S(qubits 1..%d | %d..%d) = %.3f bits\n", k, k+1, Q, _ent(ψ,Q,k)); end

## §7 — Pushing to the breaking point with a *truly global* entangled state

Everything so far stayed cheap because the states, though non-Clifford, were **low-rank**: the GHZ and
GHZ-dressed circuits carry only ~1 bit of entanglement across a cut (Schmidt rank 2), so χ barely grew.
To find where the method actually *breaks*, we need a circuit whose entanglement **grows with `Q`** — a
genuinely global, high-rank state.

First, pick that circuit by its **exact** mid-cut entanglement entropy (statevector only, no MPS — so
this is cheap and settles "is the target really globally entangled?" before we spend any MPS effort):

Measured exact mid-cut entropy `S_mid` (bits):

| circuit | Q=4 | Q=6 | Q=8 | Q=10 | Q=12 | Q=14 | Q=16 |
|---|---|---|---|---|---|---|---|
| fixed-depth brick (L=2) | 1.60 | 1.00 | 1.60 | 1.00 | 1.60 | 1.00 | 1.60 |
| **longrange** | 0.60 | 1.93 | 2.09 | 3.18 | 3.48 | 4.40 | **4.79** |

A fixed-depth brickwork *saturates* (its entanglement is capped by the light-cone, independent of `Q`)
— no good. The **`longrange`** circuit — H·T on all, a CNOT chain, a layer of CNOTs reaching across the
midpoint `i→i+Q/2`, then more T — has mid-cut entropy that **climbs steadily with `Q`** (well past GHZ's
flat 1 bit), while staying constant-depth so the teleportation count is only ~`3Q`. That is our stress
test: a global, growing-entanglement, non-Clifford state.

In [ ]:
# matrix-free exact statevector (qubit i = MSB bit Q-i) — for entropy & validation at large Q
const _Hs=ComplexF64[1 1;1 -1]/sqrt(2); const _Ts=ComplexF64[1 0;0 exp(im*π/4)]
const _Ss=ComplexF64[1 0;0 im]; const _Xs=ComplexF64[0 1;1 0]; const _Ys=ComplexF64[0 -im;im 0]; const _Zs=ComplexF64[1 0;0 -1]
function _ap1!(ψ,U,i,Q)
    step=1<<(Q-i)
    @inbounds for base in 0:(2step):(1<<Q-1), off in 0:(step-1)
        i0=base+off+1; i1=i0+step; a,c=ψ[i0],ψ[i1]; ψ[i0]=U[1,1]a+U[1,2]c; ψ[i1]=U[2,1]a+U[2,2]c
    end
end
function _apcx!(ψ,ct,tg,Q)
    bc=Q-ct; bt=Q-tg
    @inbounds for idx in 0:(1<<Q-1)
        if ((idx>>bc)&1)==1 && ((idx>>bt)&1)==0; j=idx|(1<<bt); ψ[idx+1],ψ[j+1]=ψ[j+1],ψ[idx+1]; end
    end
end
function exact_sv(Q,circ)
    ψ=zeros(ComplexF64,1<<Q); ψ[1]=1
    for g in circ
        g[1]===:H && _ap1!(ψ,_Hs,g[2],Q); g[1]===:T && _ap1!(ψ,_Ts,g[2],Q)
        g[1]===:S && _ap1!(ψ,_Ss,g[2],Q); g[1]===:CNOT && _apcx!(ψ,g[2],g[3],Q)
    end; ψ
end
Smid(ψ,Q)=(k=Q÷2; s=svdvals(reshape(ψ,1<<(Q-k),1<<k)); p=s.^2; p=p[p.>1e-14]; -sum(p.*log2.(p)))
using LinearAlgebra

longrange(Q)=begin
    c=Any[]; for i in 1:Q; push!(c,(:H,i)); end; for i in 1:Q; push!(c,(:T,i)); end
    for i in 1:(Q-1); push!(c,(:CNOT,i,i+1)); end
    for i in 1:(Q÷2); push!(c,(:CNOT,i,i+Q÷2)); end
    for i in 1:Q; push!(c,(:T,i)); end
    for i in 1:(Q-1); push!(c,(:CNOT,i,i+1)); end; for i in 1:Q; push!(c,(:T,i)); end; c
end
brickL2(Q)=begin
    c=Any[]; for i in 1:Q; push!(c,(:H,i)); end
    for _ in 1:2
        for i in 1:2:(Q-1); push!(c,(:CNOT,i,i+1)); end; for i in 1:Q; push!(c,(:T,i)); end
        for i in 2:2:(Q-1); push!(c,(:CNOT,i,i+1)); end; for i in 1:Q; push!(c,(:T,i)); end
    end; c
end
println("exact mid-cut entropy S_mid (bits):")
for (nm,f) in (("brick L=2",brickL2), ("longrange",longrange))
    print("  ", rpad(nm,11)); for Q in 4:2:16; @printf("Q%d:%.2f ",Q,Smid(exact_sv(Q,f(Q)),Q)); end; println()
end

Now run the **MPS** on `longrange` with *generous* bond dimension (`maxdim = 2048`, effectively
uncapped for these `Q`) and watch two things grow: the **peak χ the state actually demands** (with the
cap clear, this is the true bond dimension) and the **runtime**. Because cost is `O(χ³)` per gate over
an `O(Q)`-patch chain, once χ climbs into the hundreds-to-thousands the wall is a *time/memory* wall.

Measured (uncapped `maxdim=2048`, exact to ~10⁻¹⁴):

| Q | peak χ | S_mid | time |
|---|---|---|---|
| 4 | 128 | 0.60 | 74 s |
| 6 | 256 | 1.93 | 113 s |

Two points already pin the law `χ ≈ 75·2^{S_mid}` (75·2^0.60 ≈ 114 ≈ 128; 75·2^1.93 ≈ 286 ≈ 256). Feeding
in the exact `S_mid(Q)` from §7's calibration extrapolates the true bond dimension the state demands:

| Q | 8 | 10 | 12 | 14 | 16 |
|---|---|---|---|---|---|
| predicted peak χ | ~320 | ~680 | ~840 | ~1600 | ~2100 |

Since cost is `O(χ³)`, this is a steep wall: `Q = 8` already pushes a single uncapped run into *many
minutes* (empirically it did not finish inside a 30-minute budget while sharing the machine), and by
`Q ≈ 14–16` the demanded `χ ≈ 1600–2100` turns one circuit into *hours* on a single core. **That is
where a truly global non-Clifford register breaks this method on one machine.**

The mechanism is exactly the bond-dimension law of `example_bond_dimension_study.ipynb`, now driven by
*qubit count* instead of distance. The peak χ tracks the entanglement as `χ ≈ 64 · 2^{S_mid}` (the
per-patch d=3 encoding, ~64, times `2^{logical entropy}`); since §7's calibration showed `S_mid` grows
~linearly with `Q`, **χ grows exponentially in `Q`**, and runtime with its cube. Extrapolating the fit,
`Q ≈ 14–16` already demands `χ ≈ 1500–1800` — an `O(χ³)` cost that turns a single circuit into hours on
one core. That is where a *truly global* non-Clifford register breaks this method.

Two caveats sharpen "where", both inherited from the bond-dimension study:
- **A fixed budget breaks *earlier* than the peak-χ curve suggests**, because a run needs headroom above
  the *transient* χ, not just the final one (§2 there): a `maxdim = 128` cap already spoils even `Q = 4`
  — whose *final* χ is 128 — with `O(0.1)` error, since intermediate steps transiently want more.
- **At extreme under-resolution the failure is not graceful.** As the d=7 magic run showed, once the cap
  sits far below what the state needs the norm can collapse and the simulation returns **NaN**, not a
  wrong-but-finite number.

Throughout, the union-find decoder and the logical bookkeeping never falter — it is always the bond
dimension of the entangled state that gives.

In [ ]:
set_precision!(cutoff=1e-6, maxdim=2048)       # generous → measure the TRUE peak χ the state needs
println("Q  gates  χpeak  cap_hits  S_mid  maxErr     time(s)")
for Q in (4, 6)                                # extend to 8,10,12 to watch χ (and time) explode — Q≥8 is many minutes
    circ = longrange(Q); c = build_code(3; npatch=Q+1); Random.seed!(7)
    set_precision!(cutoff=1e-6, maxdim=2048)
    t = @elapsed M = run_circuit_mq(c, Q, circ; ec=false, R=3, C=2, B=2, use_AD=true)
    ψ = exact_sv(Q, circ)
    specs = Vector{Any}[]
    for i in 1:Q; push!(specs,[(i,:Z)]); push!(specs,[(i,:X)]); end
    push!(specs,[(i,:X) for i in 1:Q])
    pexp(s) = (φ=copy(ψ); for (i,P) in s; _ap1!(φ, P===:X ? _Xs : P===:Y ? _Ys : _Zs, i, Q); end; real(ψ'φ))
    err = maximum(abs(logical_expect(M,s) - pexp(s)) for s in specs)
    @printf("%-2d %-5d  %-5d  %-8d  %.2f   %.2e   %5.1f\n", Q, length(circ), CHIMAX[], CAP_HITS[], Smid(ψ,Q), err, t)
end

## §8 — What we learned, and where the boundary is

**Correctness is not the limit.** Across all three circuits — the T-GHZ (§4), the centre-crossing
entangler (§4b), and the deeper GHZ-dressed magic circuit (§5) — every measured correlator tracks the
exact `2^Q` state vector to ~10⁻¹⁴, and the §6 observables confirm the engine produces a genuine,
certifiably non-Clifford magic state (irrational `(1±1/√2)/8` basis probabilities), not a Clifford
stand-in. The generalisation was purely in the driver; the physics (transversal gates, teleportation,
frame propagation, decoding) was already qubit-agnostic.

**The number of qubits, by itself, is *not* the limit.** The T-GHZ sweep (§4) reached Q=5 with the bond
dimension pinned at χ=32 and runtime growing only linearly — Q=10+ of that kind would be routine. What
costs is the **entanglement a circuit spreads across the side-by-side chain**, which the two circuits
isolate cleanly:

- **Low-rank states are free in `Q`.** A GHZ has Schmidt rank 2 across any cut, so χ is set by each
  patch's own d=3 encoding (χ=32) no matter how many patches — flat χ, linear time (§4).
- **Entanglement across the chain is the exponential cost.** The centre-crossing circuit (§4b) and the
  truly-global `longrange` stress test (§7) both grow their mid-cut entanglement with `Q`, and χ climbs
  as `χ ≈ 75·2^{S_mid}` — the same exponential wall the distance axis hits, now driven by circuit
  structure. **Measured/extrapolated, that wall lands at `Q ≈ 14–16`** (χ ≈ 1600–2100, hours per
  circuit) for a globally-entangled non-Clifford register on one machine — with `Q = 8` already many
  minutes uncapped.

So "how many logical qubits" is the wrong question; the right one is "how much entanglement, and laid
out how". **Levers if more entangled qubits are the goal:** (a) **lattice surgery** instead of
transversal CNOTs — local merges do not span the chain, so distant-qubit entanglement is far cheaper for
the MPS (`5.Lattice-Surgery` already has the primitives); (b) a qubit ordering that keeps interacting
patches adjacent; (c) the `maxdim`/`CAP_HITS` convergence discipline from
`example_bond_dimension_study.ipynb`. As on the distance axis, the honest boundary is the **bond
dimension of the entangled state** — and it is now measured, not guessed.